# Module 4: Transformer Models
Fine-tuning LLM based on Transformers architecture (Causal LM on ruGPT-3).

> Run cells strictly sequentially from top to bottom!

In [ ]:
# ----- Cell 1: Environment Setup & Imports -----
# IMPORTANT: disable TensorFlow/Keras inside transformers (we use only PyTorch)
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device in use: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ----- Cell 2: Load Data -----
df_train = pd.read_csv('data/train_sentences.csv')
dataset = Dataset.from_pandas(df_train)
print(f"Dataset loaded. Number of sentences: {len(dataset)}")

In [ ]:
# ----- Cell 3: Load ruGPT-3 Model -----
model_name = 'sberbank-ai/rugpt3small_based_on_gpt2'

print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_name)
model = model.to(device)

# GPT does not have a pad_token by default
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model deployed on: {next(model.parameters()).device}")

In [ ]:
# ----- Cell 4: Dataset Tokenization -----
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['sentence'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )
    # For Causal LM, labels are equal to input_ids
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['sentence']
)
tokenized_dataset.set_format('torch')
print(f"Done! Number of examples: {len(tokenized_dataset)}")

In [ ]:
# ----- Cell 5: Fine-Tuning -----
training_args = TrainingArguments(
    output_dir='./results_rugpt3',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    warmup_steps=50,
    weight_decay=0.01,
    optim='adamw_torch',
    fp16=(device == 'cuda'),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Starting fine-tuning...")
trainer.train()
print("Training completed!")

In [ ]:
# ----- Cell 6: Save Model -----
save_path = './abai_rugpt3_final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

In [ ]:
# ----- Cell 7: Style Generation -----
from transformers import pipeline

generator = pipeline(
    'text-generation',
    model='./abai_rugpt3_final',
    tokenizer='./abai_rugpt3_final',
    device=0 if device == 'cuda' else -1
)

prompt = "Адам баласы"
output = generator(
    prompt,
    max_new_tokens=80,
    num_return_sequences=3,
    temperature=0.9,
    do_sample=True,
    top_k=50,
    top_p=0.92,
    pad_token_id=tokenizer.eos_token_id
)

print(f"\n=== Text Generation (Seed: '{prompt}') ===\n")
for i, result in enumerate(output):
    print(f"--- Option {i+1} ---")
    print(result['generated_text'])
    print()